<a href="https://colab.research.google.com/github/dsdsgege/vision-robustness-analyzing/blob/tamas_dev/Tamas_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from IPython.testing import test
import numpy as np
import torch
from torchvision import datasets
from torchvision.transforms import v2 as transforms
from torch.utils.data import DataLoader, random_split

In [8]:
class DataManager:
    def __init__(self, dataset_name='mnist', data_dir='./data', batch_size=100, valid_split=0.2, test_split=0.1, num_workers=2):
        """
        Inicializálja az adatkezelő osztályt.
        """
        self.dataset_name = dataset_name.lower()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.valid_split = valid_split
        self.test_split = test_split
        self.num_workers = num_workers

        # Adathalmazok leképezése
        self.datasets_dict = {
            'mnist': datasets.MNIST,
            'cifar10': datasets.CIFAR10,
            'cifar100': datasets.CIFAR100
        }

        if self.dataset_name not in self.datasets_dict:
            raise ValueError(f"Nem támogatott adathalmaz: {self.dataset_name}")

        self.DatasetClass = self.datasets_dict[self.dataset_name]

    def _get_mean_std(self):
        """Kiszámolja az adathalmaz átlagát és szórását."""
        print(f"{self.dataset_name} statisztikák számítása...")
        temp_transform = transforms.Compose([transforms.ToImage(), transforms.ToDtype(torch.float32, scale=True)])
        temp_set = self.DatasetClass(root=self.data_dir, train=True, download=True, transform=temp_transform)

        # MNIST esetén (N, H, W), CIFAR esetén (N, H, W, C)
        loader = DataLoader(temp_set, batch_size=1024)
        all_data = next(iter(loader))[0]
        mean = all_data.mean().item()
        std = all_data.std().item()
        return [mean], [std]

    def get_loaders(self):
        """Létrehozza a DataLoader objektumokat."""
        mean, std = self._get_mean_std()

        transform_pipeline = transforms.Compose([
            transforms.ToImage(),
            transforms.ToDtype(torch.float32, scale=True),
            transforms.Normalize(mean=mean, std=std)
        ])

        full_train_dataset = self.DatasetClass(root=self.data_dir, train=True, download=True, transform=transform_pipeline)
        test_dataset_full = self.DatasetClass(root=self.data_dir, train=False, download=True, transform=transform_pipeline)

        total_train_size = len(full_train_dataset)
        valid_size = int(total_train_size * self.valid_split)
        train_size = total_train_size - valid_size

        train_dataset, valid_dataset = random_split(full_train_dataset, [train_size, valid_size])

        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
        valid_loader = DataLoader(valid_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)
        test_loader = DataLoader(test_dataset_full, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

        return train_loader, valid_loader, test_loader

In [9]:
# Adatok beolvasása MNIST-tel a szerverhibák elkerülése végett
data_manager = DataManager(dataset_name='mnist', batch_size=100)

train_loader, valid_loader, test_loader = data_manager.get_loaders()

print(f"Tanító halmaz mérete: {len(train_loader.dataset)}")
print(f"Validációs halmaz mérete: {len(valid_loader.dataset)}")
print(f"Teszt halmaz mérete: {len(test_loader.dataset)}")

mnist statisztikák számítása...


100%|██████████| 9.91M/9.91M [00:00<00:00, 15.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 489kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.61MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.4MB/s]


Tanító halmaz mérete: 48000
Validációs halmaz mérete: 12000
Teszt halmaz mérete: 10000
